# Dataset validation

Run this notebook first. It mounts Google Drive, checks the expected YOLO directory layout, and reports image/label counts before training.

In [ ]:
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Not running in Colab; using the configured path if it exists.")

DATASET_DIR = Path("/content/drive/MyDrive/indonesia-license-plate-model/dataset")
if not DATASET_DIR.exists():
    raise FileNotFoundError(
        f"{DATASET_DIR} was not found. Upload the dataset to Google Drive or update DATASET_DIR."
    )

image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
for split in ("train", "val", "test"):
    image_dir = DATASET_DIR / "images" / split
    label_dir = DATASET_DIR / "labels" / split
    image_count = sum(1 for p in image_dir.rglob("*") if p.suffix.lower() in image_extensions) if image_dir.exists() else 0
    label_count = len(list(label_dir.rglob("*.txt"))) if label_dir.exists() else 0
    print(f"{split:5} images={image_count:6} labels={label_count:6}  image_dir={image_dir}")
    if split != "test" and image_count == 0:
        raise ValueError(f"No {split} images found under {image_dir}")

In [ ]:
from collections import Counter

bad_rows = []
class_ids = Counter()
for label_path in (DATASET_DIR / "labels").rglob("*.txt"):
    for line_number, line in enumerate(label_path.read_text().splitlines(), start=1):
        fields = line.split()
        if len(fields) != 5:
            bad_rows.append((label_path, line_number, "expected 5 fields"))
            continue
        try:
            class_id, *coordinates = map(float, fields)
        except ValueError:
            bad_rows.append((label_path, line_number, "non-numeric value"))
            continue
        if int(class_id) != class_id or any(value < 0 or value > 1 for value in coordinates):
            bad_rows.append((label_path, line_number, "class or normalized coordinate out of range"))
        class_ids[int(class_id)] += 1

print("Objects by class:", dict(class_ids))
if bad_rows:
    print("Invalid label rows (first 10):")
    for row in bad_rows[:10]:
        print(row)
    raise ValueError(f"Found {len(bad_rows)} invalid label rows")
print("All YOLO label rows look valid.")

Expected layout:

    dataset/
    ├── images/{train,val,test}/
    └── labels/{train,val,test}/

If the test split is absent, the training and evaluation notebooks can still use the validation split.